# 05 - Model Selection

This notebook compares the logistic-regression baseline, the Optuna-tuned XGBoost candidate from notebook 03, and the PyTorch/CUDA TabFM candidate from notebook 04a_tabFM_modeling.ipynb. Average precision (PR AUC) is the primary selection metric because churn is the minority class; ROC AUC, recall, precision, and F1 provide complementary context. The untouched test partition is evaluated only after model comparison.

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
import plotly.graph_objects as go
import torch
from dotenv import load_dotenv
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tabfm import TabFMClassifier, tabfm_v1_0_0_pytorch as tabfm_v1_0_0
from xgboost import XGBClassifier

from churn_ml.models.evaluate_model import classification_metrics

RANDOM_STATE = 42
TARGET_COLUMN = "Churn Value"
CHURN_THRESHOLD = None  # Set a value from 0 to 1 to override the training churn-rate threshold.
TABFM_N_ESTIMATORS = 8
TABFM_BATCH_SIZE = 1
TABFM_MAX_CONTEXT_ROWS = 2048
TABFM_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def find_project_file(relative_path):
    for directory in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = directory / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {relative_path} from {Path.cwd()} or its parent directories.")

def positive_class_probability(classifier, X):
    class_labels = list(classifier.classes_)
    if 1 not in class_labels:
        raise ValueError(f"Expected positive class label 1 in {class_labels}.")
    return classifier.predict_proba(X)[:, class_labels.index(1)]

def threshold_metrics(y_true, y_proba, threshold):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "pr_auc": average_precision_score(y_true, y_proba),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

LOGISTIC_DATA_PATH = find_project_file(Path("data/processed/prediction_df_logistic_regression.csv"))
XGBOOST_DATA_PATH = find_project_file(Path("data/processed/prediction_df_xgboost.csv"))
TABFM_DATA_PATH = XGBOOST_DATA_PATH
VALUE_DATA_PATH = find_project_file(Path("data/raw/Telco_customer_churn.csv"))
PROJECT_ROOT = LOGISTIC_DATA_PATH.parents[2]
load_dotenv(PROJECT_ROOT / ".env", override=False)
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
HF_TOKEN_CONFIGURED = bool(os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN"))
MLFLOW_EXPERIMENT_NAME = "telco-churn-modeling"
mlflow.set_tracking_uri("sqlite:///" + (PROJECT_ROOT / "mlflow.db").as_posix())
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
TABFM_REPO_ID = "google/tabfm-1.0.0-pytorch"
TABFM_MODEL_TYPE = "classification"
TABFM_CHECKPOINT_DIR = PROJECT_ROOT / "artifacts/models/tabfm_checkpoint" / TABFM_MODEL_TYPE
TABFM_CHECKPOINT_PATH = TABFM_CHECKPOINT_DIR / "pytorch_model.bin"

def prepare_tabfm_checkpoint(model_type=TABFM_MODEL_TYPE):
    checkpoint_dir = PROJECT_ROOT / "artifacts/models/tabfm_checkpoint" / model_type
    checkpoint_path = checkpoint_dir / "pytorch_model.bin"
    if checkpoint_path.exists():
        print(f"Using existing converted TabFM checkpoint: {checkpoint_path}")
        return checkpoint_path

    from huggingface_hub import snapshot_download
    from safetensors.torch import load_file

    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN") or None
    snapshot_path = Path(snapshot_download(
        repo_id=TABFM_REPO_ID,
        allow_patterns=[f"{model_type}/model.safetensors", f"{model_type}/config.json", "config.json"],
        token=token,
    ))
    safetensors_path = snapshot_path / model_type / "model.safetensors"
    if not safetensors_path.exists():
        raise FileNotFoundError(f"TabFM safetensors checkpoint not found at: {safetensors_path}")

    state_dict = load_file(str(safetensors_path), device="cpu")
    torch.save(state_dict, checkpoint_path)
    print(f"Converted TabFM safetensors checkpoint to: {checkpoint_path}")
    return checkpoint_path

OPTUNA_PARAMS_PATH = PROJECT_ROOT / "artifacts/models/xgboost_optuna_best_params.json"
if not OPTUNA_PARAMS_PATH.exists():
    raise FileNotFoundError(
        f"Tuned XGBoost parameters not found at {OPTUNA_PARAMS_PATH}. Run 03_xgboost_modeling.ipynb first."
    )

logistic_model_df = pd.read_csv(LOGISTIC_DATA_PATH)
xgboost_model_df = pd.read_csv(XGBOOST_DATA_PATH)
tabfm_model_df = pd.read_csv(TABFM_DATA_PATH)
value_df = pd.read_csv(VALUE_DATA_PATH, usecols=["CustomerID", TARGET_COLUMN, "CLTV"])
assert logistic_model_df[TARGET_COLUMN].equals(xgboost_model_df[TARGET_COLUMN])
assert logistic_model_df[TARGET_COLUMN].equals(tabfm_model_df[TARGET_COLUMN])
if len(value_df) != len(logistic_model_df) or not value_df[TARGET_COLUMN].equals(logistic_model_df[TARGET_COLUMN]):
    raise ValueError("Raw CLTV values are not aligned with the processed modeling data.")

y = logistic_model_df[TARGET_COLUMN]
train_index, test_index = train_test_split(
    logistic_model_df.index, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
y_train, y_test = y.loc[train_index], y.loc[test_index]
ltv_test = value_df.loc[test_index, "CLTV"].rename("predicted_ltv_if_retained")
customer_id_test = value_df.loc[test_index, "CustomerID"]
X_train_by_model = {
    "logistic_regression": logistic_model_df.drop(columns=TARGET_COLUMN).loc[train_index],
    "xgboost": xgboost_model_df.drop(columns=TARGET_COLUMN).loc[train_index],
    "tabfm": tabfm_model_df.drop(columns=TARGET_COLUMN).loc[train_index],
}
X_test_by_model = {
    "logistic_regression": logistic_model_df.drop(columns=TARGET_COLUMN).loc[test_index],
    "xgboost": xgboost_model_df.drop(columns=TARGET_COLUMN).loc[test_index],
    "tabfm": tabfm_model_df.drop(columns=TARGET_COLUMN).loc[test_index],
}
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
with OPTUNA_PARAMS_PATH.open("r", encoding="utf-8") as file:
    tuned_xgb_config = json.load(file)
tuned_weight_multiplier = tuned_xgb_config.pop("scale_pos_weight_multiplier")
tuned_xgb_cv_pr_auc = tuned_xgb_config.pop("cv_pr_auc", None)  # Metadata, not an XGBoost parameter.
decision_threshold = float(y_train.mean()) if CHURN_THRESHOLD is None else float(CHURN_THRESHOLD)
if not 0 < decision_threshold < 1:
    raise ValueError("CHURN_THRESHOLD must be between 0 and 1.")

print(f"Prediction threshold: {decision_threshold:.1%}")
print(f"Loaded Optuna-tuned XGBoost configuration from {OPTUNA_PARAMS_PATH}")
if tuned_xgb_cv_pr_auc is not None:
    print(f"Notebook 03 Optuna CV PR AUC: {tuned_xgb_cv_pr_auc:.4f}")
print(f"TabFM device: {TABFM_DEVICE}")
print(f"Hugging Face token configured: {HF_TOKEN_CONFIGURED}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Prediction threshold: 26.5%
Loaded Optuna-tuned XGBoost configuration from W:\Workstation ExtDrive\007 Data Science\001 Data Science Training\2026_016 ML Churn Model End to End\artifacts\models\xgboost_optuna_best_params.json
Notebook 03 Optuna CV PR AUC: 0.6956
TabFM device: cuda
Hugging Face token configured: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## Candidate pipelines

The logistic pipeline is the interpretable benchmark. XGBoost is the nonlinear candidate using the locked Optuna configuration produced by notebook 03. TabFM is the zero-shot tabular foundation model candidate using the PyTorch backend from notebook 04a_tabFM_modeling.ipynb; `HF_TOKEN` is loaded from the project-root `.env` before the checkpoint request if present. The safetensors checkpoint is converted once to the `.bin` format expected by the installed TabFM loader. Imputation and scaling are inside the logistic pipeline so each fold learns them only from its own training data.

The selection notebook uses the stronger PyTorch/CUDA TabFM settings selected from notebook 04a: `TABFM_N_ESTIMATORS = 8`, `TABFM_BATCH_SIZE = 1`, and `TABFM_MAX_CONTEXT_ROWS = 2048`. This makes the TabFM comparison more representative, but it also makes cross-validation substantially slower than the logistic-regression and XGBoost candidates.

In [2]:
def build_logistic_regression():
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(max_iter=1_000, class_weight="balanced", random_state=RANDOM_STATE)),
        ]
    )

def build_xgboost():
    return XGBClassifier(
        **tuned_xgb_config,
        scale_pos_weight=scale_pos_weight * tuned_weight_multiplier,
        random_state=RANDOM_STATE,
        n_jobs=1,
        tree_method="hist",
    )

def build_tabfm():
    if TABFM_DEVICE == "cuda":
        torch.cuda.empty_cache()
    tabfm_checkpoint_path = prepare_tabfm_checkpoint()
    tabfm_backbone = tabfm_v1_0_0.load(checkpoint_path=str(tabfm_checkpoint_path), model_type=TABFM_MODEL_TYPE, device=TABFM_DEVICE)
    return TabFMClassifier(
        model=tabfm_backbone,
        n_estimators=TABFM_N_ESTIMATORS,
        batch_size=TABFM_BATCH_SIZE,
        max_num_rows=TABFM_MAX_CONTEXT_ROWS,
        random_state=RANDOM_STATE,
        use_amp=(TABFM_DEVICE == "cuda"),
        verbose=False,
    )

model_factories = {
    "logistic_regression": build_logistic_regression,
    "xgboost": build_xgboost,
    "tabfm": build_tabfm,
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## Cross-validated comparison and holdout evaluation

Select the highest mean cross-validated PR AUC. The XGBoost hyperparameters are locked from notebook 03, and TabFM uses its pretrained PyTorch checkpoint without project-specific tuning. The holdout partition remains untouched by tuning and selection. If performance is similar, prefer the simpler model unless TabFM or XGBoost has a meaningful business-value advantage.

In [3]:
cv_rows = []
for model_name, factory in model_factories.items():
    fold_rows = []
    for fold, (fold_train_index, fold_valid_index) in enumerate(cv.split(X_train_by_model[model_name], y_train), start=1):
        X_fold_train = X_train_by_model[model_name].iloc[fold_train_index]
        X_fold_valid = X_train_by_model[model_name].iloc[fold_valid_index]
        y_fold_train = y_train.iloc[fold_train_index]
        y_fold_valid = y_train.iloc[fold_valid_index]

        model = factory()
        model.fit(X_fold_train, y_fold_train)
        if model_name == "tabfm":
            fold_proba = positive_class_probability(model, X_fold_valid)
        else:
            fold_proba = model.predict_proba(X_fold_valid)[:, 1]
        fold_metrics = threshold_metrics(y_fold_valid, fold_proba, decision_threshold)
        fold_metrics["fold"] = fold
        fold_rows.append(fold_metrics)
        print(f"{model_name} fold {fold}: PR AUC = {fold_metrics['pr_auc']:.4f}")

    fold_df = pd.DataFrame(fold_rows)
    cv_rows.append({
        "model": model_name,
        **{f"mean_{metric}": fold_df[metric].mean() for metric in ["pr_auc", "roc_auc", "precision", "recall", "f1"]},
        **{f"std_{metric}": fold_df[metric].std() for metric in ["pr_auc", "roc_auc", "precision", "recall", "f1"]},
    })

cv_results = pd.DataFrame(cv_rows).sort_values("mean_pr_auc", ascending=False).set_index("model")
display(cv_results)
display(pd.DataFrame([
    {
        "candidate": "xgboost",
        "tuning_source": "03_xgboost_modeling.ipynb",
        "optuna_cv_pr_auc": tuned_xgb_cv_pr_auc,
        "eval_metric": tuned_xgb_config["eval_metric"],
        "n_estimators": tuned_xgb_config["n_estimators"],
    },
    {
        "candidate": "tabfm",
        "tuning_source": "04a_tabFM_modeling.ipynb",
        "checkpoint": TABFM_REPO_ID,
        "checkpoint_path": str(TABFM_CHECKPOINT_PATH),
        "backend": "pytorch",
        "n_estimators": TABFM_N_ESTIMATORS,
    },
]))

selected_model_name = cv_results.index[0]
holdout_rows = []
fitted_models = {}
test_probabilities = {}
for model_name, factory in model_factories.items():
    fitted_model = factory()
    fitted_model.fit(X_train_by_model[model_name], y_train)
    if model_name == "tabfm":
        test_proba = positive_class_probability(fitted_model, X_test_by_model[model_name])
    else:
        test_proba = fitted_model.predict_proba(X_test_by_model[model_name])[:, 1]
    test_pred = (test_proba >= decision_threshold).astype(int)
    holdout_rows.append({"model": model_name, **classification_metrics(y_test, test_pred, test_proba)})
    fitted_models[model_name] = fitted_model
    test_probabilities[model_name] = test_proba

holdout_metrics = pd.DataFrame(holdout_rows).set_index("model")
print(f"Selected model by mean CV PR AUC: {selected_model_name}")
display(holdout_metrics)

test_proba = test_probabilities[selected_model_name]
test_pred = (test_proba >= decision_threshold).astype(int)
confusion = confusion_matrix(y_test, test_pred, labels=[0, 1])
fig = go.Figure(go.Heatmap(
    z=confusion, x=["Predicted: no churn", "Predicted: churn"],
    y=["Actual: no churn", "Actual: churn"],
    colorscale="Blues", text=confusion, texttemplate="%{text}",
    colorbar={"title": "Customers"},
))
fig.update_layout(title=f"{selected_model_name} Confusion Matrix (threshold = {decision_threshold:.1%})")
fig.show()

logistic_regression fold 1: PR AUC = 0.6621
logistic_regression fold 2: PR AUC = 0.6561
logistic_regression fold 3: PR AUC = 0.6919
logistic_regression fold 4: PR AUC = 0.7027
logistic_regression fold 5: PR AUC = 0.6800
xgboost fold 1: PR AUC = 0.6786
xgboost fold 2: PR AUC = 0.6645
xgboost fold 3: PR AUC = 0.7181
xgboost fold 4: PR AUC = 0.7031
xgboost fold 5: PR AUC = 0.7016
Using existing converted TabFM checkpoint: W:\Workstation ExtDrive\007 Data Science\001 Data Science Training\2026_016 ML Churn Model End to End\artifacts\models\tabfm_checkpoint\classification\pytorch_model.bin
tabfm fold 1: PR AUC = 0.6783
Using existing converted TabFM checkpoint: W:\Workstation ExtDrive\007 Data Science\001 Data Science Training\2026_016 ML Churn Model End to End\artifacts\models\tabfm_checkpoint\classification\pytorch_model.bin
tabfm fold 2: PR AUC = 0.6755
Using existing converted TabFM checkpoint: W:\Workstation ExtDrive\007 Data Science\001 Data Science Training\2026_016 ML Churn Model En

,mean_pr_auc,mean_roc_auc,mean_precision,mean_recall,mean_f1,std_pr_auc,std_roc_auc,std_precision,std_recall,std_f1
model,,,,,,,,,,
tabfm,0.697127,0.866490,0.550888,0.800669,0.652550,0.019812,0.013559,0.022582,0.030798,0.023561
xgboost,0.693172,0.864844,0.426343,0.941806,0.586851,0.021351,0.012431,0.007202,0.017634,0.004047
logistic_regression,0.678589,0.857506,0.429117,0.927759,0.586767,0.019629,0.014205,0.010519,0.019871,0.012485


,candidate,tuning_source,optuna_cv_pr_auc,eval_metric,n_estimators,checkpoint,checkpoint_path,backend
0,xgboost,03_xgboost_modeling.ipynb,0.695612,aucpr,179,NaN,NaN,NaN
1,tabfm,04a_tabFM_modeling.ipynb,NaN,NaN,8,google/tabfm-1.0.0-pytorch,W:\Workstation ExtDrive\007 Data Science\001 D...,pytorch


Using existing converted TabFM checkpoint: W:\Workstation ExtDrive\007 Data Science\001 Data Science Training\2026_016 ML Churn Model End to End\artifacts\models\tabfm_checkpoint\classification\pytorch_model.bin
Selected model by mean CV PR AUC: tabfm


,accuracy,precision,recall,f1,pr_auc,roc_auc
model,,,,,,
logistic_regression,0.647977,0.425245,0.927807,0.583193,0.639283,0.846438
xgboost,0.645848,0.424970,0.946524,0.586578,0.672541,0.855912
tabfm,0.762952,0.536101,0.794118,0.640086,0.678069,0.859865


## Expected-value comparison for retention targeting

Model quality alone does not determine which customers should receive a retention offer. This comparison retrieves holdout-test `CLTV` from the raw data and treats it as predicted lifetime value if the customer is retained. CLTV is not used as a churn-model feature.

For each model, the top 100 customers are ranked by expected net value:

$$P(\text{churn}) \times 10\% \times \text{predicted LTV if retained} - \$20 - (40\% \times \$500)$$

The scenario assumes $20 outreach cost for every target, a $500 offer, and a 40% acceptance rate among targeted customers, including customers who would have stayed without intervention. The expected offer cost is therefore $200 per target. The 10% retention uplift represents the share of would-be churners saved by outreach. The uplift and acceptance rate are business assumptions, not estimates from the churn models.

In [4]:
OUTREACH_COST = 20
OFFER_COST = 500
OFFER_ACCEPTANCE_RATE = 0.40
RETENTION_UPLIFT = 0.10
TARGET_COUNT = 100

targeting_rows = []
target_ids_by_model = {}
for model_name, test_proba in test_probabilities.items():
    candidates = pd.DataFrame({
        "CustomerID": customer_id_test.to_numpy(),
        "predicted_churn_probability": test_proba,
        "predicted_ltv_if_retained": ltv_test.to_numpy(),
    })
    candidates["expected_value_before_cost"] = (
        candidates["predicted_churn_probability"]
        * RETENTION_UPLIFT
        * candidates["predicted_ltv_if_retained"]
    )
    candidates["expected_offer_cost"] = OFFER_COST * OFFER_ACCEPTANCE_RATE
    candidates["campaign_cost"] = OUTREACH_COST + candidates["expected_offer_cost"]
    candidates["expected_net_value"] = (
        candidates["expected_value_before_cost"] - candidates["campaign_cost"]
    )
    top_targets = candidates.nlargest(TARGET_COUNT, "expected_net_value")
    target_ids_by_model[model_name] = set(top_targets["CustomerID"])
    targeting_rows.append({
        "model": model_name,
        "customers_targeted": len(top_targets),
        "expected_value_before_cost": top_targets["expected_value_before_cost"].sum(),
        "outreach_cost": OUTREACH_COST * len(top_targets),
        "expected_offer_cost": top_targets["expected_offer_cost"].sum(),
        "campaign_cost": top_targets["campaign_cost"].sum(),
        "expected_net_value": top_targets["expected_net_value"].sum(),
    })

targeting_results = (
    pd.DataFrame(targeting_rows)
    .sort_values("expected_net_value", ascending=False)
    .set_index("model")
)
display(targeting_results.style.format({
    "expected_value_before_cost": "${:,.2f}",
    "outreach_cost": "${:,.2f}",
    "expected_offer_cost": "${:,.2f}",
    "campaign_cost": "${:,.2f}",
    "expected_net_value": "${:,.2f}",
}))

shared_targets = len(set.intersection(*target_ids_by_model.values()))
print(f"Shared customers in all top-{TARGET_COUNT} target lists: {shared_targets}")
pairwise_overlap = pd.DataFrame(
    [
        {
            "model_a": model_a,
            "model_b": model_b,
            "shared_top_targets": len(target_ids_by_model[model_a] & target_ids_by_model[model_b]),
        }
        for i, model_a in enumerate(target_ids_by_model)
        for model_b in list(target_ids_by_model)[i + 1:]
    ]
)
display(pairwise_overlap)

,customers_targeted,expected_value_before_cost,outreach_cost,expected_offer_cost,campaign_cost,expected_net_value
model,,,,,,
xgboost,100,"$46,647.51","$2,000.00","$20,000.00","$22,000.00","$24,647.51"
logistic_regression,100,"$45,555.11","$2,000.00","$20,000.00","$22,000.00","$23,555.11"
tabfm,100,"$38,017.86","$2,000.00","$20,000.00","$22,000.00","$16,017.86"


Shared customers in all top-100 target lists: 70


,model_a,model_b,shared_top_targets
0,logistic_regression,xgboost,86
1,logistic_regression,tabfm,73
2,xgboost,tabfm,75


## MLflow tracking

This section records the lightweight selection evidence: cross-validated comparison, holdout metrics for all candidates, selected model, targeting value comparison, and top-target overlap. It does not log model objects or TabFM weights.

In [5]:
with mlflow.start_run(run_name="05_model_selection"):
    mlflow.set_tags({
        "notebook": "05_model_selection.ipynb",
        "stage": "model_selection",
        "selection_metric": "mean_cv_pr_auc",
    })
    mlflow.log_params({
        "random_state": RANDOM_STATE,
        "target_column": TARGET_COLUMN,
        "threshold_policy": "training_churn_rate" if CHURN_THRESHOLD is None else "manual",
        "decision_threshold": decision_threshold,
        "selected_model": selected_model_name,
        "cv_splits": cv.get_n_splits(),
        "tabfm_backend": "pytorch",
        "tabfm_device": TABFM_DEVICE,
        "tabfm_n_estimators": TABFM_N_ESTIMATORS,
        "tabfm_batch_size": TABFM_BATCH_SIZE,
        "xgboost_tuning_source": "03_xgboost_modeling.ipynb",
        "tabfm_tuning_source": "04a_tabFM_modeling.ipynb",
    })
    if torch.cuda.is_available():
        mlflow.log_param("gpu_name", torch.cuda.get_device_name(0))
    for model_name, row in cv_results.iterrows():
        for metric_name, value in row.items():
            mlflow.log_metric(f"cv_{model_name}_{metric_name}", float(value))
    for model_name, row in holdout_metrics.iterrows():
        for metric_name, value in row.items():
            mlflow.log_metric(f"holdout_{model_name}_{metric_name}", float(value))
    for model_name, row in targeting_results.iterrows():
        mlflow.log_metric(f"targeting_{model_name}_expected_net_value", float(row["expected_net_value"]))
        mlflow.log_metric(f"targeting_{model_name}_campaign_cost", float(row["campaign_cost"]))
    for _, row in pairwise_overlap.iterrows():
        mlflow.log_metric(
            f"target_overlap_{row['model_a']}_vs_{row['model_b']}",
            int(row["shared_top_targets"]),
        )
    mlflow.log_metric("target_overlap_all_models", int(shared_targets))
    mlflow.log_table(cv_results.reset_index(), "tables/cv_results.json")
    mlflow.log_table(holdout_metrics.reset_index(), "tables/holdout_metrics.json")
    mlflow.log_table(targeting_results.reset_index(), "tables/targeting_results.json")
    mlflow.log_table(pairwise_overlap, "tables/pairwise_target_overlap.json")

print(f"Logged MLflow run to {PROJECT_ROOT / 'mlflow.db'}")

Logged MLflow run to W:\Workstation ExtDrive\007 Data Science\001 Data Science Training\2026_016 ML Churn Model End to End\mlflow.db


## Final selection insights

With the stronger PyTorch/CUDA TabFM settings (`TABFM_N_ESTIMATORS = 8`, `TABFM_BATCH_SIZE = 1`, `TABFM_MAX_CONTEXT_ROWS = 2048`), TabFM is the best model by mean cross-validated PR AUC: 0.6971 versus 0.6932 for XGBoost and 0.6786 for logistic regression. The TabFM advantage over XGBoost is still small relative to fold-to-fold variation, so this should be read as a narrow predictive edge rather than a decisive win.

On the untouched holdout set, TabFM is also strongest on several standard classification metrics: PR AUC 0.6781, ROC AUC 0.8599, accuracy 0.7630, precision 0.5361, and F1 0.6401. XGBoost remains close on PR AUC at 0.6725 and ROC AUC at 0.8559, and it has the highest recall at 0.9465 versus 0.7941 for TabFM. In plain modeling terms, TabFM is slightly better at ranking and precision-oriented classification, while XGBoost is more aggressive about finding churners at the current churn-rate threshold.

For the retention campaign objective, XGBoost is still the best practical choice in this saved run. The top-100 expected net value is $24,647.51 for XGBoost, $23,555.11 for logistic regression, and $16,017.86 for TabFM. This can look counterintuitive because TabFM has better aggregate predictive metrics, but the expected-value calculation is not a generic classification score. It ranks customers by predicted churn probability multiplied by retained-LTV value, then subtracts the same campaign cost assumptions. TabFM's top-100 list is therefore selecting a different mix of customers whose predicted churn-and-value combination is less profitable under the current cost model.

The target overlap reinforces that explanation. Logistic regression and XGBoost share 86 of the top 100 targets, while XGBoost and TabFM share 75, logistic regression and TabFM share 73, and only 70 customers appear in all three lists. TabFM is not merely a slightly different scorer on the same campaign list; it is changing a meaningful share of the outreach population, and that change reduces expected net value here.

The practical recommendation is to select XGBoost for the current retention-targeting workflow. TabFM is a useful benchmark and now has the strongest conventional predictive metrics, but it took much longer to run and did not convert that extra runtime into better business value. The notebook does not currently log wall-clock runtimes as metrics, so the runtime conclusion is based on observed execution rather than a saved timing table; add explicit timing if runtime should become part of the formal selection record. Under the current assumptions, XGBoost gives the best balance of expected net value, speed, operational simplicity, and predictive performance.
